# Forecasting de demanda eléctrica — explicación del proyecto, celda por celda

Este notebook explica **todo** el proyecto: qué hace cada módulo, **por qué** se
tomó cada decisión, qué problemas reales aparecieron y cómo se resolvieron.
Está pensado para leerse de arriba a abajo, ejecutando las celdas de demostración.

**Cómo está organizado**
- Cada módulo tiene: una explicación (el *qué* y el *porqué*), su código real, y
  cuando aplica, una **demo ejecutable** sobre los datos reales del repo.
- Las **decisiones críticas** y los **problemas reales** están marcados con 🔑 y 🐛.

**Requisitos para ejecutarlo**
- Ábrelo con el entorno del proyecto (`.venv`) como kernel. Si te falta Jupyter:
  `pip install jupyter ipykernel` dentro del `.venv`, o ábrelo en VS Code y elige
  el intérprete `.venv`.
- Los datos (`data/`) ya están versionados en el repo, así que las demos corren tal cual.


## 1. ¿Qué construimos y por qué?

Un **servicio que pronostica la demanda eléctrica horaria** de la región PJM
(este de EE.UU.) para las próximas 24 horas, usando datos públicos de la **EIA**.

Pero lo importante no es el modelo. Lo importante es que es un **sistema vivo**:
cada día, solo, (1) trae datos nuevos, (2) pronostica, (3) compara la predicción
de ayer contra lo que realmente pasó, y (4) actualiza métricas en un dashboard.

**El porqué (enfoque ML Engineer).** Un data scientist entrega un notebook con un
modelo. Un ML Engineer entrega un *sistema* que se mantiene y se monitorea solo.
Esa diferencia —el *loop* automatizado, versionado y con un backtest honesto— es
lo que este proyecto busca demostrar. Los modelos son sólidos, pero son lo de menos.

**El diferenciador concreto:** el historial de commits de la carpeta `data/`
(generado por un robot en la nube) es la prueba de que el loop corrió día tras día.


## 2. Arquitectura general

```
  EIA API (demanda horaria)
        |  ingesta diaria (GitHub Actions cron)
        v
  Histórico versionado en el repo (Parquet)
        |
        +--> Forecast 24h (modelos) --> Predicciones guardadas (con corte as-of)
        |                                      |
        v                                      v
  Backtest continuo  <-----------------  Valor real del día siguiente
        |
        v
  Métricas + error rodante  -->  Dashboard (Streamlit)
```

**Mapa de módulos** (`src/forecasting/`), cada uno con una sola responsabilidad:

| Módulo | Responsabilidad |
|---|---|
| `config` | Constantes (región, rutas, schema) |
| `store` | Lee/escribe el histórico con disciplina de snapshot |
| `ingest` | Trae datos de la API de la EIA |
| `bootstrap` | Backfill inicial del histórico |
| `series` | Limpia y regulariza la serie (huecos, outliers) |
| `models/` | Los 4 modelos + un registry |
| `forecast_daily` | Orquesta el forecast (corte as-of, persiste) |
| `predictions` | Guarda predicciones (idempotente) |
| `backtest` | Evalúa: forward-test e histórico |
| `metrics` | MAE y sMAPE |
| `views` | Prepara datos para el dashboard |
| `run_daily` | Pipeline diario (ingesta + forecast) |


In [ ]:
# Setup: ubicarnos en la raíz del repo y poder importar el paquete
import os, sys, pathlib

# Jupyter abre el notebook con CWD = carpeta del notebook (notebooks/).
# Subimos a la raíz del repo para que las rutas de data/ resuelvan.
if not pathlib.Path('data/demand_history.parquet').exists() and pathlib.Path('../data/demand_history.parquet').exists():
    os.chdir('..')

# Fallback por si el paquete no está instalado con `pip install -e .`
src = pathlib.Path.cwd() / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
from forecasting import store
from forecasting.config import DEMAND_HISTORY_PATH

hist = store.read_demand_history(DEMAND_HISTORY_PATH)
print('Filas:', len(hist), '| rango:', hist['period'].min(), '->', hist['period'].max())
hist.head()

# Parte 1 — Fundación de datos

Antes de cualquier modelo necesitamos **traer los datos y guardarlos bien**. Aquí
viven las decisiones más sutiles del proyecto (y dos bugs reales que solo aparecen
con datos de verdad).

### `config.py` — constantes

Sin lógica: solo configuración compartida. Tener un único lugar para la región, las
rutas y el *schema* evita repetir strings mágicos por todo el código. Fíjate en el
schema del histórico: por cada hora guardamos **dos** valores de demanda
(`value_first_reported` y `value_current`) — el porqué se explica en `store`.

```python
# src/forecasting/config.py
"""Constantes compartidas. Sin lógica: solo configuración."""
from pathlib import Path

# Región objetivo (balancing authority de la EIA). Ver spec §3.
REGION = "PJM"

# Type codes de la EIA en el endpoint region-data:
#   "D"  = demanda real (lo que predecimos/verificamos)
#   "DF" = pronóstico día-adelante de la propia EIA (baseline futuro, roadmap)
DEMAND_TYPE = "D"

EIA_BASE_URL = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

# Rutas de datos versionados.
DATA_DIR = Path("data")
DEMAND_HISTORY_PATH = DATA_DIR / "demand_history.parquet"
PREDICTIONS_PATH = DATA_DIR / "predictions.parquet"

# Schema del histórico de demanda. Todas las horas en UTC.
DEMAND_COLUMNS = [
    "period",                 # datetime64[ns, UTC] — hora de la observación
    "respondent",             # str — región (p.ej. "PJM")
    "value_first_reported",   # float — primer valor publicado por la EIA (MWh)
    "value_current",          # float — valor tras posibles revisiones (MWh)
    "first_seen_at",          # datetime64[ns, UTC] — cuándo lo vimos por 1a vez
    "last_updated_at",        # datetime64[ns, UTC] — última vez que cambió
]

```

### `store.py` — el histórico y la disciplina de snapshot

🔑 **Decisión crítica #1: guardar dos versiones de cada valor real.**

La EIA **revisa sus cifras hacia atrás**: el valor de demanda de una hora puede
cambiar días después. Si guardáramos solo "el valor", nuestro backtest mentiría:
compararíamos la predicción de ayer contra un número que *hoy* es distinto al que
existía ayer. Eso es una forma sutil de *data leakage* (usar información del futuro).

La solución: por cada `(hora, región)` guardamos
- `value_first_reported`: el primer valor que publicó la EIA, y
- `value_current`: el valor tras posibles revisiones.

`upsert_demand` implementa esto de forma **idempotente** (re-ingerir lo mismo no
duplica filas) y robusta (maneja NaN y duplicados dentro del mismo lote). El
almacenamiento es **Parquet** (columnar, comprime bien, tipado) commiteado al repo:
ese archivo versionado *es* la prueba del loop vivo.

```python
# src/forecasting/store.py
"""Lectura/escritura del histórico de demanda con disciplina de snapshot."""
import math
from pathlib import Path

import pandas as pd

from forecasting.config import DEMAND_COLUMNS


def _values_equal(a: float, b: float) -> bool:
    """Compara valores tolerando NaN y ruido de punto flotante."""
    if math.isnan(a) and math.isnan(b):
        return True          # seguía faltando → no es revisión
    if math.isnan(a) or math.isnan(b):
        return False         # cambió de/a NaN → sí es revisión
    return math.isclose(a, b, rel_tol=1e-9)


def _empty_history() -> pd.DataFrame:
    """DataFrame vacío con el schema correcto (tipos incluidos)."""
    df = pd.DataFrame(columns=DEMAND_COLUMNS)
    df["period"] = pd.to_datetime(df["period"], utc=True)
    df["first_seen_at"] = pd.to_datetime(df["first_seen_at"], utc=True)
    df["last_updated_at"] = pd.to_datetime(df["last_updated_at"], utc=True)
    df["value_first_reported"] = df["value_first_reported"].astype("float64")
    df["value_current"] = df["value_current"].astype("float64")
    df["respondent"] = df["respondent"].astype("object")
    return df


def read_demand_history(path: Path) -> pd.DataFrame:
    """Lee el histórico desde Parquet. Si no existe, devuelve vacío con schema."""
    path = Path(path)
    if not path.exists():
        return _empty_history()
    return pd.read_parquet(path)


def upsert_demand(path: Path, observations: pd.DataFrame, now: pd.Timestamp) -> dict:
    """Funde observaciones crudas en el histórico con rastreo de revisiones.

    `observations` debe tener columnas: period (UTC), respondent, value.
    `now` es el timestamp UTC del momento de ingesta (inyectado para testeo).
    Devuelve conteos: {"inserted", "revised", "unchanged"}.
    """
    path = Path(path)
    observations = observations.drop_duplicates(subset=["period", "respondent"], keep="last")
    history = read_demand_history(path)
    existing = {
        (r.period, r.respondent): r for r in history.itertuples(index=False)
    }

    rows = []
    counts = {"inserted": 0, "revised": 0, "unchanged": 0}
    for obs in observations.itertuples(index=False):
        key = (obs.period, obs.respondent)
        prev = existing.pop(key, None)
        if prev is None:
            counts["inserted"] += 1
            rows.append(
                {
                    "period": obs.period,
                    "respondent": obs.respondent,
                    "value_first_reported": float(obs.value),
                    "value_current": float(obs.value),
                    "first_seen_at": now,
                    "last_updated_at": now,
                }
            )
        elif not _values_equal(float(obs.value), prev.value_current):
            counts["revised"] += 1
            rows.append(
                {
                    "period": prev.period,
                    "respondent": prev.respondent,
                    "value_first_reported": prev.value_first_reported,
                    "value_current": float(obs.value),
                    "first_seen_at": prev.first_seen_at,
                    "last_updated_at": now,
                }
            )
        else:
            counts["unchanged"] += 1
            rows.append(prev._asdict())

    # Preservar filas existentes que no vinieron en el batch de observaciones
    for prev in existing.values():
        rows.append(prev._asdict())

    merged = pd.DataFrame(rows, columns=DEMAND_COLUMNS).sort_values(
        ["respondent", "period"]
    ).reset_index(drop=True)

    path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_parquet(path, index=False)
    return counts

```

In [ ]:
# DEMO: cómo upsert_demand distingue insertar / revisar / sin-cambio
import tempfile, pathlib
from forecasting import store

tmp = pathlib.Path(tempfile.mkdtemp()) / 'demo.parquet'
now1 = pd.Timestamp('2025-01-01T00:00', tz='UTC')

obs = pd.DataFrame({
    'period': pd.to_datetime(['2025-01-01T00', '2025-01-01T01'], utc=True),
    'respondent': ['PJM', 'PJM'],
    'value': [100.0, 110.0],
})
print('1) inserta:', store.upsert_demand(tmp, obs, now=now1))
print('2) idéntico (idempotente):', store.upsert_demand(tmp, obs, now=now1))

# la EIA 'revisa' la primera hora: 100 -> 105
rev = obs.copy(); rev.loc[0, 'value'] = 105.0
now2 = pd.Timestamp('2025-01-02T00:00', tz='UTC')
print('3) revisa una:', store.upsert_demand(tmp, rev, now=now2))

out = store.read_demand_history(tmp).sort_values('period')
print('\nfirst_reported se conserva, current se actualiza:')
out[['period', 'value_first_reported', 'value_current']]

### `ingest.py` — traer datos de la EIA

Separa **parsear** (puro, sin red) de **traer** (HTTP). Esa separación hace que el
parseo se pueda testear sin internet. `fetch_demand` arma la URL de la API v2 de la
EIA, **pagina** (la API devuelve máx. 5000 filas por llamada) y normaliza la
respuesta a las columnas que `store` espera. La API key va en la URL (requisito de
la EIA) y se lee de una variable de entorno, nunca hardcodeada.

```python
# src/forecasting/ingest.py
"""Ingesta de demanda horaria desde la API v2 de la EIA."""
import pandas as pd
import requests

from forecasting.config import DEMAND_TYPE, EIA_BASE_URL

PARSED_COLUMNS = ["period", "respondent", "value"]


def parse_eia_response(payload: dict) -> pd.DataFrame:
    """Normaliza el JSON de la EIA a un DataFrame (period UTC, respondent, value).

    La EIA devuelve `period` como "YYYY-MM-DDTHH" (UTC) y `value` como string.
    """
    records = payload.get("response", {}).get("data", [])
    if not records:
        return (
            pd.DataFrame({c: pd.Series(dtype="object") for c in PARSED_COLUMNS})
            .astype({"value": "float64"})
            .assign(period=lambda d: pd.to_datetime(d["period"], utc=True))
            .astype({"period": "datetime64[ns, UTC]"})
        )

    df = pd.DataFrame(records)
    out = pd.DataFrame(
        {
            # pandas 2+ infiere resolución "us" por defecto; forzamos "ns" para
            # coincidir con el dtype que usa store.upsert_demand en el histórico.
            "period": pd.to_datetime(df["period"], utc=True, format="%Y-%m-%dT%H").astype(
                "datetime64[ns, UTC]"
            ),
            "respondent": df["respondent"].astype("object"),
            "value": df["value"].astype("float64"),
        }
    )
    return out[PARSED_COLUMNS]


def fetch_demand(
    respondent: str,
    start: str,
    end: str,
    api_key: str,
    page_length: int = 5000,
) -> pd.DataFrame:
    """Trae demanda horaria (type=D) de la EIA para [start, end], paginando.

    start/end en formato "YYYY-MM-DDTHH" (UTC). Devuelve DataFrame normalizado.
    """
    frames = []
    offset = 0
    while True:
        params = {
            "api_key": api_key,
            "frequency": "hourly",
            "data[0]": "value",
            "facets[respondent][]": respondent,
            "facets[type][]": DEMAND_TYPE,
            "start": start,
            "end": end,
            "sort[0][column]": "period",
            "sort[0][direction]": "asc",
            "offset": offset,
            "length": page_length,
        }
        resp = requests.get(EIA_BASE_URL, params=params, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        frames.append(parse_eia_response(payload))

        total = int(payload.get("response", {}).get("total", 0))
        offset += page_length
        if offset >= total:
            break

    return pd.concat(frames, ignore_index=True)

```

### `bootstrap.py` — backfill inicial

🔑 **Decisión: resolver el "arranque en frío".** El día 1 no tenemos datos, y
LightGBM/SARIMAX no pueden entrenar sin historia. `bootstrap` trae varios años de
una sola vez (aquí: 2021–2026, ~48k horas) para que los modelos tengan contexto y
el dashboard no nazca vacío. Nota que `fetch_fn` es inyectable: en los tests le
pasamos una función falsa para no tocar la red.

```python
# src/forecasting/bootstrap.py
"""Backfill único del histórico de demanda (datos viejos para entrenar/backtest)."""
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from forecasting import ingest, store
from forecasting.config import DEMAND_HISTORY_PATH, REGION


def backfill_demand(
    path: Path,
    respondent: str,
    start: str,
    end: str,
    api_key: str,
    now: pd.Timestamp,
    fetch_fn=ingest.fetch_demand,
) -> dict:
    """Trae demanda en [start, end] y la persiste vía upsert. `fetch_fn` inyectable."""
    observations = fetch_fn(
        respondent=respondent, start=start, end=end, api_key=api_key
    )
    return store.upsert_demand(path, observations, now=now)


if __name__ == "__main__":
    # Uso: python -m forecasting.bootstrap 2021-01-01T00 2026-06-01T00
    import sys

    load_dotenv()
    api_key = os.environ["EIA_API_KEY"]
    start, end = sys.argv[1], sys.argv[2]
    counts = backfill_demand(
        path=DEMAND_HISTORY_PATH,
        respondent=REGION,
        start=start,
        end=end,
        api_key=api_key,
        now=pd.Timestamp.now(tz="UTC"),
    )
    print(f"Backfill {REGION} [{start}..{end}]: {counts}")

```

### `series.py` — limpieza y regularización (aquí vivieron 2 bugs reales)

🐛 **Problema real #1: huecos.** Los datos de la EIA tienen **horas faltantes**
(p.ej. en cambios de horario). Los modelos asumen una serie horaria *contigua*: el
seasonal naive usa *posiciones* ("hace 168 posiciones") y LightGBM busca el lag por
*timestamp exacto*. Un hueco → LightGBM lanzaba `KeyError` y el naive quedaba
desalineado (su "hace una semana" ya no eran 168 horas reales).

🐛 **Problema real #2: valores basura.** La EIA devuelve ~72 horas con **centinelas**
(≈ 2.147e9, que es 2³¹−1, el típico "missing" codificado como entero máximo) y
algunos ceros. Esto rompía LightGBM (entrenaba con basura: MAE de ~48% vs ~5% real)
y además corrompía la *evaluación* (un "valor real" de 2.1e9 infla el error de todos
los modelos).

**Cómo se descubrió:** al verificar sobre datos reales, LightGBM daba un MAE
absurdamente alto frente al baseline. Eso *no tiene sentido* para demanda eléctrica,
así que se investigó (debugging sistemático) y apareció el `max` de la serie en
2.1e9. Moraleja: **compilar y pasar tests sintéticos ≠ funcionar con datos reales**.

**La solución (ambos problemas, un solo lugar):** `regularize_hourly` marca como
faltantes los valores implausibles (`<= 0` o `> 5×` la mediana robusta), reindexa a
una grilla horaria completa, e interpola. Devuelve una serie limpia y contigua,
lista para modelar y evaluar. La fuente de verdad (el Parquet crudo) no se toca.

```python
# src/forecasting/series.py
"""Limpieza y regularización de la serie temporal antes de alimentar a los modelos.

Los datos de la EIA del mundo real traen dos problemas:
  1. Huecos: horas faltantes (p.ej. en cambios de horario). Los modelos asumen una
     serie horaria contigua (el seasonal naive usa posiciones; LightGBM busca lags
     por timestamp exacto), así que un hueco rompe o desalinea.
  2. Valores basura: centinelas (~2^31 ≈ 2.1e9 que la EIA usa como "missing"),
     ceros y outliers absurdos que corrompen tanto el entrenamiento como la
     evaluación (un "valor real" de 2.1e9 infla el error de todos los modelos).

`regularize_hourly` resuelve ambos: marca como faltantes los valores implausibles,
reindexa a una grilla horaria completa, e interpola. Devuelve una serie limpia,
contigua y lista para modelar/evaluar.
"""
import pandas as pd

# Un valor se considera implausible (centinela/outlier) si es <= 0 o si supera
# este múltiplo de la mediana robusta. La demanda eléctrica vive en una banda
# estrecha y positiva, así que un múltiplo generoso (5x) descarta basura
# (millones/miles de millones) sin tocar los picos reales.
MAX_MEDIAN_RATIO = 5.0


def regularize_hourly(df: pd.DataFrame, value_col: str = "value") -> pd.DataFrame:
    """Limpia outliers, reindexa a grilla horaria UTC completa e interpola huecos.

    Devuelve un DataFrame con columnas [period, value_col] contiguo por hora y
    sin valores implausibles.
    """
    if df.empty:
        return df[["period", value_col]]

    s = df.set_index("period")[value_col].sort_index()

    # 1. Marca valores implausibles como faltantes (NaN) usando la mediana robusta.
    positive = s[s > 0]
    if not positive.empty:
        med = positive.median()
        implausible = (s <= 0) | (s > MAX_MEDIAN_RATIO * med)
        s = s.mask(implausible)

    # 2. Reindexa a la grilla horaria completa (rellena huecos como NaN).
    full = pd.date_range(s.index.min(), s.index.max(), freq="h")  # hereda el tz UTC

    # 3. Interpola por tiempo; ffill/bfill cubre cualquier NaN en los extremos.
    s = s.reindex(full).interpolate(method="time", limit_direction="both").ffill().bfill()

    out = s.reset_index()
    out.columns = ["period", value_col]
    return out

```

In [ ]:
# DEMO: regularize_hourly arregla un hueco y un valor centinela
from forecasting import series

idx = pd.to_datetime(['2025-01-01T00','2025-01-01T01','2025-01-01T03','2025-01-01T04'], utc=True)
#                                            ^ falta 02:00 (hueco)
gappy = pd.DataFrame({'period': idx, 'value': [100.0, 110.0, 2_147_480_000.0, 130.0]})
#                                                        ^ valor basura (centinela)
print('Entrada (con hueco y basura):')
print(gappy.to_string(index=False))
print('\nSalida regularizada (hueco rellenado, basura interpolada):')
series.regularize_hourly(gappy)

# Parte 2 — Los modelos

🔑 **Decisión crítica #2: una interfaz común.** Todo modelo es una clase con un
atributo `name` y un método `predict(history, horizon) -> DataFrame`. Gracias a
esa interfaz uniforme, los modelos son **intercambiables** y se pueden comparar de
forma justa con el mismo código. `history` es una serie limpia `(period, value)` y
la salida tiene columnas `period / model / prediction` (los probabilísticos añaden
`q10`/`q90`).

Comparamos **cuatro niveles** de sofisticación. La pregunta interesante: ¿le gana
un modelo de 2024 a un LightGBM bien hecho y a un baseline ingenuo, medido
honestamente? La respuesta es interesante sea cual sea — y demuestra criterio.

| Nivel | Modelo | Qué demuestra |
|---|---|---|
| 0 | Seasonal naive | La vara mínima honesta |
| 1 | SARIMAX + Fourier | Econometría clásica (job semanal aislado) |
| 2 | LightGBM | ML tabular de industria |
| 3 | Chronos-Bolt | Foundation model de series, zero-shot |

### `models/base.py` — el contrato común

Define las columnas de salida y `future_periods` (las próximas N horas tras el
histórico), que todos los modelos reutilizan. Un detalle: como la última hora del
histórico es *tz-aware* (UTC), el rango de horas futuras hereda el timezone solo.

```python
# src/forecasting/models/base.py
"""Contrato común de los modelos de forecasting y helpers compartidos."""
from __future__ import annotations

import pandas as pd

# Columnas mínimas que todo modelo devuelve. Los probabilísticos pueden añadir más.
FORECAST_COLUMNS = ["period", "model", "prediction"]


def future_periods(history: pd.DataFrame, horizon: int) -> pd.DatetimeIndex:
    """Las próximas `horizon` horas (UTC) después de la última del histórico."""
    last = history["period"].iloc[-1]
    return pd.date_range(
        start=last + pd.Timedelta(hours=1), periods=horizon, freq="h"
    )

```

### Nivel 0 — `naive.py` (seasonal naive)

"Mañana a las 3pm será como hace exactamente una semana a las 3pm." Suena tonto,
pero en series muy estacionales es **difícil de superar**. Se incluye a propósito:
muchos proyectos presumen un modelo complejo sin demostrar que le gana a esto.

```python
# src/forecasting/models/naive.py
"""Seasonal naive: el valor de hace exactamente una semana (168 h)."""
import pandas as pd

from forecasting.models.base import FORECAST_COLUMNS, future_periods

WEEK_HOURS = 168


class SeasonalNaive:
    name = "seasonal_naive"

    def predict(self, history: pd.DataFrame, horizon: int) -> pd.DataFrame:
        if len(history) < WEEK_HOURS:
            raise ValueError(
                f"seasonal_naive necesita >= {WEEK_HOURS} h de histórico, "
                f"recibió {len(history)}"
            )
        values = history["value"].to_numpy()
        n = len(values)
        preds = [values[n - WEEK_HOURS + i] for i in range(horizon)]
        return pd.DataFrame(
            {
                "period": future_periods(history, horizon),
                "model": self.name,
                "prediction": preds,
            }
        )[FORECAST_COLUMNS]

```

In [ ]:
# DEMO: el naive sobre los datos reales (forecast de las próximas 24h)
from forecasting import series
from forecasting.models.naive import SeasonalNaive

serie = series.regularize_hourly(pd.DataFrame({'period': hist['period'], 'value': hist['value_current']}))
fc_naive = SeasonalNaive().predict(serie, horizon=24)
fc_naive.head()

### Nivel 2 — `lightgbm_model.py`

🔑 **Decisión: forecasting directo, sin recursión.** Las features son el calendario
de la hora objetivo (hora, día de semana, mes, fin de semana) más dos *lags* que ya
conocemos: el de hace 24h y el de hace 168h. Para un horizonte de 24h, esos lags de
las horas futuras **ya están en el histórico**, así que no hace falta predecir de
forma recursiva (que acumularía error). Es lo que se usa de verdad en industria.

```python
# src/forecasting/models/lightgbm_model.py
"""LightGBM con features de calendario + lags conocidos (24 h y 168 h)."""
import lightgbm as lgb
import pandas as pd

from forecasting.models.base import FORECAST_COLUMNS, future_periods

LAG_DAY = 24
LAG_WEEK = 168


def _calendar(idx: pd.DatetimeIndex) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "hour": idx.hour,
            "dayofweek": idx.dayofweek,
            "month": idx.month,
            "is_weekend": (idx.dayofweek >= 5).astype(int),
        }
    )


class LightGBMForecaster:
    name = "lightgbm"

    def __init__(self, **params):
        self.params = {
            "n_estimators": 200,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "random_state": 0,
            "verbose": -1,
            **params,
        }

    def _build_features(self, periods: pd.DatetimeIndex, lookup: dict) -> pd.DataFrame:
        feats = _calendar(periods)
        feats["lag_24"] = [lookup[p - pd.Timedelta(hours=LAG_DAY)] for p in periods]
        feats["lag_168"] = [lookup[p - pd.Timedelta(hours=LAG_WEEK)] for p in periods]
        return feats

    def predict(self, history: pd.DataFrame, horizon: int) -> pd.DataFrame:
        if horizon > LAG_DAY:
            raise ValueError(
                f"lightgbm asume horizon <= {LAG_DAY} (lag_24 conocido sin recursión)"
            )
        if len(history) < LAG_WEEK + 1:
            raise ValueError(
                f"lightgbm necesita > {LAG_WEEK} h de histórico, recibió {len(history)}"
            )
        series = history.set_index("period")["value"]
        lookup = series.to_dict()

        train_idx = series.index[LAG_WEEK:]
        X_train = self._build_features(train_idx, lookup)
        y_train = series.loc[train_idx].to_numpy()

        model = lgb.LGBMRegressor(**self.params)
        model.fit(X_train, y_train)

        future = future_periods(history, horizon)
        X_future = self._build_features(future, lookup)
        preds = model.predict(X_future)

        return pd.DataFrame(
            {"period": future, "model": self.name, "prediction": preds}
        )[FORECAST_COLUMNS]

```

In [ ]:
# DEMO: LightGBM sobre datos reales
from forecasting.models.lightgbm_model import LightGBMForecaster
fc_lgbm = LightGBMForecaster().predict(serie, horizon=24)
fc_lgbm.head()

### Nivel 1 — `sarimax.py`

🔑 **Decisión crítica #3: Fourier en vez de SARIMA "puro".** SARIMA clásico maneja
*una sola* estacionalidad, pero nuestra serie tiene tres (diaria=24h, semanal=168h,
anual). Forzar `m=168` es inviable (no converge, lentísimo). La forma canónica
(*dynamic harmonic regression*, de Hyndman) es modelar las estacionalidades con
**términos de Fourier** (senos/cosenos) como variables exógenas, y dejar que el
ARIMA modele solo la autocorrelación de corto plazo. Usamos `statsforecast` (rápido,
pensado para correr desatendido). Por su costo, SARIMAX corre en un **job semanal
aislado**, no en el diario: si no converge, no tumba el loop principal.

```python
# src/forecasting/models/sarimax.py
"""SARIMAX vía dynamic harmonic regression: Fourier (24h,168h) como exógenas + AutoARIMA."""
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

from forecasting.models.base import FORECAST_COLUMNS, future_periods

SEASONAL_PERIODS = (24, 168)
N_HARMONICS = (3, 3)


def _fourier_terms(periods: pd.DatetimeIndex, origin: pd.Timestamp) -> pd.DataFrame:
    hours = ((periods - origin) / pd.Timedelta(hours=1)).to_numpy(dtype=float)
    cols = {}
    for period, k_max in zip(SEASONAL_PERIODS, N_HARMONICS):
        for k in range(1, k_max + 1):
            cols[f"sin_{period}_{k}"] = np.sin(2 * np.pi * k * hours / period)
            cols[f"cos_{period}_{k}"] = np.cos(2 * np.pi * k * hours / period)
    return pd.DataFrame(cols)


class SarimaxForecaster:
    name = "sarimax"

    def predict(self, history: pd.DataFrame, horizon: int) -> pd.DataFrame:
        origin = history["period"].iloc[0]
        hist_idx = pd.DatetimeIndex(history["period"])
        future = future_periods(history, horizon)

        X_hist = _fourier_terms(hist_idx, origin)
        X_fut = _fourier_terms(future, origin)

        train = pd.DataFrame(
            {"unique_id": "PJM", "ds": hist_idx.tz_localize(None), "y": history["value"].to_numpy()}
        )
        train = pd.concat([train, X_hist], axis=1)

        X_df = pd.DataFrame({"unique_id": "PJM", "ds": future.tz_localize(None)})
        X_df = pd.concat([X_df, X_fut], axis=1)

        sf = StatsForecast(models=[AutoARIMA(seasonal=False)], freq="h")
        fc = sf.forecast(df=train, h=horizon, X_df=X_df)

        return pd.DataFrame(
            {"period": future, "model": self.name, "prediction": fc["AutoARIMA"].to_numpy()}
        )[FORECAST_COLUMNS]

```

### Nivel 3 — `chronos_model.py` (foundation model)

🔑 **Lo "actual".** Chronos-Bolt (Amazon) es un *transformer preentrenado* en
millones de series; predice **zero-shot** (sin entrenar en tus datos) y da forecasts
**probabilísticos** (cuantiles q10/q90, no solo un punto).

**Paralelo con tu proyecto de visión:** igual que usaste una CNN preentrenada
(EfficientNet) para sacar *embeddings* de imágenes sin entrenar desde cero, aquí
usas un transformer preentrenado para forecasting. Misma idea de *transfer
learning*, dominio nuevo.

🐛 **Problema real #3 (API).** El plan asumía que el método se llamaba con
`context=...`, pero la versión instalada (chronos 2.3.0) usa `inputs=...`. Se
detectó haciendo un *spike* de verificación (probar la API real antes de escribir
el adaptador) en vez de confiar en la memoria. Moraleja: **verifica la API, no la
inventes.**

```python
# src/forecasting/models/chronos_model.py
"""Chronos-Bolt (Amazon): transformer preentrenado, forecasting zero-shot probabilístico."""
import pandas as pd
import torch
from chronos import BaseChronosPipeline

from forecasting.models.base import FORECAST_COLUMNS, future_periods

QUANTILE_LEVELS = [0.1, 0.5, 0.9]


class ChronosForecaster:
    name = "chronos"

    def __init__(self, model_id: str = "amazon/chronos-bolt-small", context_length: int = 512):
        self.model_id = model_id
        self.context_length = context_length
        self._pipeline = None  # carga perezosa: solo al primer predict

    def _load(self):
        if self._pipeline is None:
            self._pipeline = BaseChronosPipeline.from_pretrained(
                self.model_id, device_map="cpu", torch_dtype=torch.float32
            )
        return self._pipeline

    def predict(self, history: pd.DataFrame, horizon: int) -> pd.DataFrame:
        pipeline = self._load()
        values = history["value"].to_numpy()[-self.context_length:]
        context = torch.tensor(values, dtype=torch.float32)

        # Nota: la API instalada (chronos-forecasting 2.3.0) nombra el parámetro
        # `inputs`, no `context` como en versiones previas de la librería.
        # El spike de verificación confirmó esto antes de escribir el código.
        quantiles, _mean = pipeline.predict_quantiles(
            inputs=context, prediction_length=horizon, quantile_levels=QUANTILE_LEVELS
        )
        q = quantiles[0].numpy()  # forma (horizon, 3): [q10, q50, q90]

        return pd.DataFrame(
            {
                "period": future_periods(history, horizon),
                "model": self.name,
                "prediction": q[:, 1],
                "q10": q[:, 0],
                "q90": q[:, 2],
            }
        )[FORECAST_COLUMNS + ["q10", "q90"]]

```

In [ ]:
# DEMO (puede tardar ~10s la 1a vez: descarga el modelo 'tiny' de HuggingFace)
from forecasting.models.chronos_model import ChronosForecaster
fc_chronos = ChronosForecaster(model_id='amazon/chronos-bolt-tiny').predict(serie, horizon=24)
fc_chronos.head()  # fíjate en las bandas q10/q90 (incertidumbre)

### `models/__init__.py` — el registry

Agrupa los modelos por **cadencia de ejecución**: `daily_models()` (rápidos y
robustos: naive, lightgbm, chronos) corren cada día; `weekly_models()` (sarimax,
costoso) corre semanal y aislado. Esto refleja una decisión de producción: no todo
tiene que correr con la misma frecuencia.

```python
# src/forecasting/models/__init__.py
"""Registry de modelos de forecasting, agrupados por cadencia de ejecución (spec §5)."""
from forecasting.models.chronos_model import ChronosForecaster
from forecasting.models.lightgbm_model import LightGBMForecaster
from forecasting.models.naive import SeasonalNaive
from forecasting.models.sarimax import SarimaxForecaster


def daily_models():
    """Modelos rápidos/robustos que corren en el loop diario."""
    return [SeasonalNaive(), LightGBMForecaster(), ChronosForecaster()]


def weekly_models():
    """Modelos costosos, aislados en un job semanal (no tumban el loop diario)."""
    return [SarimaxForecaster()]


def all_models():
    return daily_models() + weekly_models()

```

# Parte 3 — Orquestación y backtest

Aquí se junta todo: correr los modelos día a día y medir honestamente qué tan buenos son.

### `forecast_daily.py` — el corte *as-of*

🔑 **Decisión crítica #4: anti-leakage al pronosticar.** Al predecir en un momento
`as_of`, el modelo **solo puede ver datos con `period <= as_of`**. `run_forecast`
corta el histórico ahí antes de pasarlo al modelo. Nunca se predice con información
del futuro. Además es **robusto**: si un modelo lanza una excepción, se omite y se
reporta en `failures` — un modelo roto no tumba a los demás (justo lo que pasó
cuando LightGBM fallaba con los huecos).

```python
# src/forecasting/forecast_daily.py
"""Orquestación del forecast diario: corte as-of, corre modelos, persiste predicciones."""
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from forecasting import predictions, series, store
from forecasting.config import DEMAND_HISTORY_PATH, PREDICTIONS_PATH
from forecasting.predictions import PREDICTION_COLUMNS


def _as_series(history: pd.DataFrame) -> pd.DataFrame:
    """Convierte el histórico del store a la serie limpia (period, value) que usan los modelos.

    Regulariza a grilla horaria contigua: los datos de la EIA tienen huecos que
    romperían a los modelos basados en lags/posición (ver forecasting.series).
    """
    raw = pd.DataFrame(
        {"period": history["period"], "value": history["value_current"]}
    )
    return series.regularize_hourly(raw)


def run_forecast(history: pd.DataFrame, models, as_of: pd.Timestamp, horizon: int = 24):
    """Corre cada modelo sobre el histórico cortado en as_of.

    Devuelve (predictions_df, failures) donde failures es lista de (model_name, error_str).
    Robustez: un modelo que lanza excepción se omite y se registra.
    """
    series = _as_series(history)
    sliced = series[series["period"] <= as_of].reset_index(drop=True)
    last_period = sliced["period"].iloc[-1]

    frames = []
    failures = []
    for model in models:
        try:
            fc = model.predict(sliced, horizon)
        except Exception as exc:  # noqa: BLE001 — un modelo no debe tumbar el loop
            failures.append((model.name, str(exc)))
            continue
        rows = pd.DataFrame(
            {
                "forecast_made_at": as_of,
                "model": model.name,
                "target_period": fc["period"].to_numpy(),
                "horizon": (
                    (fc["period"] - last_period) / pd.Timedelta(hours=1)
                ).astype("int64").to_numpy(),
                "prediction": fc["prediction"].to_numpy(),
                "q10": fc["q10"].to_numpy() if "q10" in fc.columns else float("nan"),
                "q90": fc["q90"].to_numpy() if "q90" in fc.columns else float("nan"),
            }
        )
        frames.append(rows)

    if frames:
        preds = pd.concat(frames, ignore_index=True)[PREDICTION_COLUMNS]
    else:
        preds = predictions._empty()
    return preds, failures


def forecast_and_store(
    history, models, as_of, horizon=24, predictions_path=PREDICTIONS_PATH
) -> dict:
    """run_forecast + persistencia. Devuelve resumen {stored, failures}."""
    preds, failures = run_forecast(history, models, as_of, horizon)
    stored = predictions.upsert_predictions(predictions_path, preds)
    return {"stored": stored, "failures": failures}


if __name__ == "__main__":
    from forecasting.models import daily_models

    load_dotenv()
    history = store.read_demand_history(DEMAND_HISTORY_PATH)
    as_of = history["period"].max()
    summary = forecast_and_store(history, daily_models(), as_of=as_of, horizon=24)
    print(f"Forecast as_of={as_of}: {summary}")

```

In [ ]:
# DEMO: el corte as-of de verdad impide ver el futuro
from forecasting import forecast_daily

class ModeloEspia:
    name = 'espia'
    def predict(self, history, horizon):
        self.max_visto = history['period'].max()  # qué tan lejos vio
        fut = pd.date_range(history['period'].iloc[-1] + pd.Timedelta(hours=1), periods=horizon, freq='h')
        return pd.DataFrame({'period': fut, 'model': self.name, 'prediction': 0.0})

espia = ModeloEspia()
as_of = hist['period'].iloc[-72]   # un punto en el pasado
preds, failures = forecast_daily.run_forecast(hist, [espia], as_of=as_of, horizon=24)
print('as_of pedido :', as_of)
print('máximo visto :', espia.max_visto, '  <- nunca pasa de as_of')
print('fallos       :', failures)
preds.head()

### `predictions.py` — guardar predicciones (idempotente)

Cada predicción se identifica por `(forecast_made_at, model, target_period)`. El
`upsert` es **re-ejecutable**: correr el mismo forecast dos veces no duplica filas.
Esto importa para un cron que podría reintentarse.

```python
# src/forecasting/predictions.py
"""Persistencia idempotente de predicciones en Parquet."""
from pathlib import Path

import pandas as pd

PREDICTION_COLUMNS = [
    "forecast_made_at",
    "model",
    "target_period",
    "horizon",
    "prediction",
    "q10",
    "q90",
]

_KEY = ["forecast_made_at", "model", "target_period"]


def _empty() -> pd.DataFrame:
    df = pd.DataFrame(columns=PREDICTION_COLUMNS)
    df["forecast_made_at"] = pd.to_datetime(df["forecast_made_at"], utc=True)
    df["target_period"] = pd.to_datetime(df["target_period"], utc=True)
    df["horizon"] = df["horizon"].astype("int64")
    for col in ("prediction", "q10", "q90"):
        df[col] = df[col].astype("float64")
    df["model"] = df["model"].astype("object")
    return df


def read_predictions(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        return _empty()
    return pd.read_parquet(path)


def upsert_predictions(path: Path, new_rows: pd.DataFrame) -> dict:
    """Funde predicciones nuevas; clave (forecast_made_at, model, target_period).

    Re-ejecutable: una predicción ya existente se reemplaza (no se duplica).
    Devuelve {"inserted", "updated"}.
    """
    path = Path(path)
    new_rows = new_rows.drop_duplicates(subset=_KEY, keep="last")[PREDICTION_COLUMNS]
    existing = read_predictions(path)

    existing_keys = set(map(tuple, existing[_KEY].to_numpy())) if len(existing) else set()
    new_keys = set(map(tuple, new_rows[_KEY].to_numpy()))
    inserted = len(new_keys - existing_keys)
    updated = len(new_keys & existing_keys)

    if len(existing):
        keep_mask = ~existing.set_index(_KEY).index.isin(new_keys)
        kept = existing[keep_mask]
    else:
        kept = existing

    merged = (
        pd.concat([kept, new_rows], ignore_index=True)
        .sort_values(_KEY)
        .reset_index(drop=True)
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_parquet(path, index=False)
    return {"inserted": inserted, "updated": updated}

```

### `metrics.py` — MAE y sMAPE

MAE da el error en MWh (interpretable); sMAPE en % (comparable). Se eligió sMAPE
sobre MAPE clásico porque **evita dividir por cero**. Detalle fino: la división se
hace con `np.divide(..., where=...)` para no generar `RuntimeWarning` cuando el
denominador es 0 — limpieza que mantiene la salida sin ruido.

```python
# src/forecasting/metrics.py
"""Métricas de error de forecasting (puras, sin estado)."""
import numpy as np


def mae(y_true, y_pred) -> float:
    """Mean Absolute Error (en las unidades de la serie, p.ej. MWh)."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(yt - yp)))


def smape(y_true, y_pred) -> float:
    """Symmetric MAPE en % (0 = perfecto). Robusto a denominador cero."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    denom = np.abs(yt) + np.abs(yp)
    # División segura: solo divide donde denom != 0; el resto queda en 0 (sin warning).
    contrib = np.zeros_like(denom)
    np.divide(2 * np.abs(yp - yt), denom, out=contrib, where=denom != 0)
    return float(np.mean(contrib) * 100)

```

### `backtest.py` — los dos regímenes de evaluación

🔑 **Decisión crítica #5: distinguir dos formas de medir, y ser honesto sobre cuál
es cuál.**

1. **Forward-test (honesto):** evalúa las predicciones que el loop guardó *en vivo*
   contra los valores reales. Es point-in-time real, pero empieza vacío y crece con
   los días.
2. **Backtest histórico (optimista):** re-corre los modelos sobre el histórico viejo
   desde varios orígenes (*rolling-origin*) y mide contra datos ya revisados. Llena
   el dashboard desde el día 1, pero es **ligeramente optimista** (el modelo "ve"
   datos más limpios de los que habría tenido en tiempo real). Lo etiquetamos como tal.

Ambos usan valores **limpios** para evaluar (recuerda el bug de los 2.1e9: un "real"
basura inflaría el error de todos).

```python
# src/forecasting/backtest.py
"""Evaluación de forecasts en dos regímenes: forward-test y backtest histórico."""
import pandas as pd

from forecasting import metrics, series


def _metrics_by_model(df: pd.DataFrame) -> pd.DataFrame:
    """df con columnas model/actual/prediction -> MAE/sMAPE/n por modelo."""
    rows = []
    for model, g in df.groupby("model"):
        rows.append(
            {
                "model": model,
                "mae": metrics.mae(g["actual"], g["prediction"]),
                "smape": metrics.smape(g["actual"], g["prediction"]),
                "n": len(g),
            }
        )
    return pd.DataFrame(rows)


def evaluate_predictions(
    predictions: pd.DataFrame, history: pd.DataFrame, actual_col: str = "value_current"
) -> pd.DataFrame:
    """Forward-test: une predicciones guardadas con los reales y mide error por modelo.

    Une por target_period == period. Inner join: predicciones sin real (futuro aún
    no observado) se descartan.
    """
    # Limpia los valores reales (centinelas/outliers de la EIA) antes de medir:
    # un "real" basura (p.ej. 2.1e9) inflaría el error de todos los modelos.
    clean = series.regularize_hourly(
        pd.DataFrame({"period": history["period"], "value": history[actual_col]})
    ).rename(columns={"period": "target_period", "value": "actual"})
    merged = predictions.merge(clean, on="target_period", how="inner")
    return _metrics_by_model(merged)


def rolling_origin_backtest(
    history: pd.DataFrame,
    models,
    origins,
    horizon: int = 24,
    actual_col: str = "value_current",
) -> pd.DataFrame:
    """Backtest histórico (optimista): para cada origin corta el histórico, corre los
    modelos y compara contra el valor real (revisado). Agrega MAE/sMAPE por modelo.
    """
    # Serie limpia y regularizada (outliers y huecos tratados): se usa tanto como
    # entrada de los modelos como para los valores reales de la evaluación, así la
    # basura de la EIA no corrompe ni el entrenamiento ni las métricas.
    model_input = series.regularize_hourly(
        pd.DataFrame({"period": history["period"], "value": history[actual_col]})
    )
    actual_lookup = dict(zip(model_input["period"], model_input["value"]))

    records = []
    for origin in origins:
        sliced = model_input[model_input["period"] <= origin].reset_index(drop=True)
        for model in models:
            try:
                fc = model.predict(sliced, horizon)
            except Exception:  # noqa: BLE001 — un modelo no debe tumbar el backtest
                continue
            for target, pred in zip(fc["period"], fc["prediction"]):
                if target in actual_lookup:
                    records.append(
                        {"model": model.name, "actual": actual_lookup[target], "prediction": pred}
                    )

    return _metrics_by_model(pd.DataFrame(records, columns=["model", "actual", "prediction"]))

```

In [ ]:
# DEMO — EL PAYOFF: ¿quién gana, medido honestamente?
# Backtest rolling-origin de naive vs LightGBM sobre 5 orígenes reales.
from forecasting import backtest
from forecasting.models.naive import SeasonalNaive
from forecasting.models.lightgbm_model import LightGBMForecaster

origins = list(hist['period'].iloc[[-24*40, -24*30, -24*20, -24*10, -48]])
tabla = backtest.rolling_origin_backtest(
    hist, [SeasonalNaive(), LightGBMForecaster()], origins, horizon=24
)
tabla.round(2)
# Esperado: LightGBM (~5% sMAPE) le gana ~2x al naive (~9-10%).
# (Antes del fix de outliers, LightGBM daba ~18% por entrenar con basura.)

# Parte 4 — Dashboard y el loop vivo (CI/CD)

Lo que vuelve el proyecto "vivo" y visible.

### `views.py` — datos para el dashboard

Funciones **puras** que preparan lo que el dashboard grafica: el forecast más
reciente por modelo, el predicho-vs-real, y el error rodante. Al ser puras, se
testean fácil. El dashboard (Streamlit) solo las llama y dibuja — lógica fuera de
la UI.

```python
# src/forecasting/views.py
"""Preparación de datos para el dashboard (funciones puras, testeables)."""
import pandas as pd


def latest_forecast(predictions: pd.DataFrame) -> pd.DataFrame:
    """Las filas del forecast más reciente por modelo (panel 'forecast actual')."""
    if predictions.empty:
        return predictions
    latest = predictions.groupby("model")["forecast_made_at"].transform("max")
    return predictions[predictions["forecast_made_at"] == latest].reset_index(drop=True)


def predicted_vs_actual(
    predictions: pd.DataFrame, history: pd.DataFrame, actual_col: str = "value_current"
) -> pd.DataFrame:
    """Une predicciones con el valor real por target_period (para graficar)."""
    actuals = history[["period", actual_col]].rename(
        columns={"period": "target_period", actual_col: "actual"}
    )
    merged = predictions.merge(actuals, on="target_period", how="inner")
    return merged[["target_period", "model", "prediction", "actual"]]


def rolling_error(predicted_vs_actual_df: pd.DataFrame, window: int = 168) -> pd.DataFrame:
    """MAE rodante por modelo a lo largo del tiempo (degradación visible)."""
    df = predicted_vs_actual_df.copy()
    df["abs_error"] = (df["actual"] - df["prediction"]).abs()
    df = df.sort_values(["model", "target_period"])
    df["rolling_mae"] = (
        df.groupby("model")["abs_error"]
        .rolling(window, min_periods=window)
        .mean()
        .reset_index(level=0, drop=True)
    )
    return df[["target_period", "model", "rolling_mae"]].reset_index(drop=True)

```

In [ ]:
# DEMO: predicho vs real, listo para graficar
from forecasting import views, predictions
from forecasting.config import PREDICTIONS_PATH
preds_all = predictions.read_predictions(PREDICTIONS_PATH)
pva = views.predicted_vs_actual(preds_all, hist)
print('filas con match predicho-vs-real:', len(pva))
pva.head()

### `dashboard/app.py` y `run_daily.py`

`app.py` es la app Streamlit: lee los Parquet versionados y muestra forecast actual,
predicho-vs-real, error rodante y las métricas de los dos regímenes. **No importa
ninguna librería pesada** (torch, lightgbm…) — solo pandas — por eso se pudo
desplegar en Streamlit Cloud (free tier). 🐛 **Problema real #4:** el primer deploy
falló porque `requirements.txt` incluía PyTorch; se separó en `requirements.txt`
(liviano, dashboard) y `requirements-models.txt` (pesado, cron/local).

`run_daily.py` es el **entrypoint del cron**: trae datos recientes, los funde al
histórico, y corre el forecast. Es lo que GitHub Actions ejecuta cada día.

```python
# src/forecasting/run_daily.py
"""Pipeline diario: ingesta de datos frescos de la EIA + forecast. Entrypoint del cron."""
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from forecasting import forecast_daily, ingest, store
from forecasting.config import DEMAND_HISTORY_PATH, PREDICTIONS_PATH, REGION
from forecasting.models import daily_models


def run_daily(
    history_path: Path,
    predictions_path: Path,
    models,
    api_key: str,
    now: pd.Timestamp,
    recent_days: int = 3,
    fetch_fn=ingest.fetch_demand,
    horizon: int = 24,
    respondent: str = REGION,
) -> dict:
    """Trae los últimos `recent_days` días, funde al histórico y corre el forecast."""
    start = (now - pd.Timedelta(days=recent_days)).strftime("%Y-%m-%dT%H")
    end = now.strftime("%Y-%m-%dT%H")
    fresh = fetch_fn(respondent=respondent, start=start, end=end, api_key=api_key)
    ingested = store.upsert_demand(history_path, fresh, now=now)

    history = store.read_demand_history(history_path)
    as_of = history["period"].max()
    forecast = forecast_daily.forecast_and_store(
        history, models, as_of=as_of, horizon=horizon, predictions_path=predictions_path
    )
    return {"ingested": ingested, "forecast": forecast}


if __name__ == "__main__":
    load_dotenv()
    api_key = os.environ["EIA_API_KEY"]
    summary = run_daily(
        history_path=DEMAND_HISTORY_PATH, predictions_path=PREDICTIONS_PATH,
        models=daily_models(), api_key=api_key, now=pd.Timestamp.now(tz="UTC"),
    )
    print(f"run_daily: {summary}")

```

### Los workflows de GitHub Actions (el loop vivo)

🔑 **Lo que hace ML-Engineer a esto.** Tres archivos YAML en `.github/workflows/`:
- `tests.yml`: corre los tests en cada push (integración continua).
- `daily.yml`: cron diario que ejecuta `run_daily` y **commitea** `data/`.
- `weekly.yml`: cron semanal para SARIMAX (aislado).

Conceptos clave que aparecen aquí:
- **cron**: `"0 12 * * *"` = todos los días a las 12:00 UTC.
- **secret**: la API key se inyecta con `${{ secrets.EIA_API_KEY }}`, nunca en el código.
- **permisos** `contents: write` y un `git commit/push` desde el robot: así el
  historial de `data/` crece solo, día a día — la prueba del sistema vivo.

Abajo el workflow diario completo:

```python
# .github/workflows/daily.yml
name: daily-forecast
on:
  schedule:
    - cron: "0 12 * * *"
  workflow_dispatch:
jobs:
  forecast:
    runs-on: ubuntu-latest
    permissions:
      contents: write
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - name: Instalar dependencias
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt -r requirements-models.txt
          pip install -e .
      - name: Ingesta + forecast
        env:
          EIA_API_KEY: ${{ secrets.EIA_API_KEY }}
        run: python -m forecasting.run_daily
      - name: Commitear datos actualizados
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add data/
          git diff --staged --quiet || git commit -m "data: actualiza histórico y forecast ($(date -u +%Y-%m-%d))"
          git push

```

# Parte 5 — Decisiones críticas y problemas (resumen)

### Decisiones críticas
| # | Decisión | Por qué |
|---|---|---|
| 1 | Guardar `first_reported` y `current` | La EIA revisa cifras; evita leakage en el backtest |
| 2 | Interfaz común de modelos | Hace los modelos intercambiables y comparables |
| 3 | Fourier en vez de SARIMA puro | Maneja estacionalidad múltiple (24h/168h/anual) |
| 4 | Corte as-of al pronosticar | Anti-leakage: el modelo nunca ve el futuro |
| 5 | Dos regímenes de backtest | Honestidad: forward-test real vs histórico optimista |
| + | Repo dedicado (no monorepo) | Los commits diarios del robot no ensucian el portafolio |
| + | Deps separadas | Dashboard liviano deployable; modelos pesados solo en cron/local |

### Problemas reales (todos hallados al verificar con datos de verdad)
| # | Problema | Solución |
|---|---|---|
| 1 | Huecos horarios en la EIA | `regularize_hourly` reindexa a grilla completa |
| 2 | Valores basura (~2.1e9, ceros) | `regularize_hourly` los marca implausibles e interpola |
| 3 | API de Chronos (`inputs` vs `context`) | *Spike* de verificación antes de implementar |
| 4 | Streamlit Cloud no instala PyTorch | Separar requirements liviano/pesado |

El hilo conductor: **compilar y pasar tests sintéticos no es lo mismo que funcionar
con datos reales.** Los problemas 1 y 2 solo aparecieron al correr el sistema de
verdad — por eso "verificar corriendo" es parte del trabajo, no un extra.

# Glosario (conceptos de software/ML engineering)

- **Data leakage:** usar, sin querer, información del futuro (o datos que no tendrías
  en tiempo real) al entrenar/evaluar. Hace que un modelo se vea mejor de lo que es.
- **Backtest rolling-origin:** evaluar un modelo de series "rebobinando" a varios
  puntos del pasado, pronosticando hacia adelante desde cada uno y promediando el error.
- **Idempotente:** una operación que, repetida, deja el mismo resultado (re-ingerir
  los mismos datos no duplica filas).
- **Transfer learning:** reutilizar un modelo ya entrenado en muchísimos datos para
  tu problema, en vez de entrenar desde cero (EfficientNet en visión, Chronos en series).
- **Foundation model:** modelo grande preentrenado de propósito general que sirve de
  base para muchas tareas (aquí, Chronos para forecasting zero-shot).
- **Zero-shot:** predecir sobre datos nuevos sin entrenar específicamente en ellos.
- **CI/CD:** Integración/Entrega Continua. Automatizar tests y despliegue ante cada
  cambio (aquí, GitHub Actions).
- **cron:** un planificador de tareas por tiempo (ej. "todos los días a medianoche").
- **secret:** un valor sensible (API key) que la plataforma inyecta de forma segura,
  fuera del código.
- **src layout / `pip install -e .`:** poner el código en `src/` e instalarlo en modo
  editable para poder importarlo (`import forecasting`) desde cualquier lugar.
- **TDD (test-driven development):** escribir primero el test que falla, luego el
  código mínimo que lo hace pasar. Así se construyó todo este proyecto.
- **as-of:** "a fecha de". El corte de datos disponibles en un instante dado.

# Cómo correr todo (resumen práctico)

```bash
# 1. entorno
python -m venv .venv && .venv\Scripts\Activate.ps1   # (Windows)
pip install -r requirements.txt -r requirements-models.txt
pip install -e .
cp .env.example .env   # y pon tu EIA_API_KEY

# 2. datos + forecast
python -m forecasting.bootstrap 2021-01-01T00 2026-06-19T23   # backfill (una vez)
python -m forecasting.run_daily                                # ingesta + forecast

# 3. dashboard local
streamlit run dashboard/app.py

# 4. tests
pytest -q                       # rápidos
pytest -q -m "slow or not slow"  # incluye SARIMAX y Chronos
```

**Cierre.** Construiste un sistema de ML Engineer completo: datos → modelos →
orquestación con backtest honesto → dashboard → CI/CD, corriendo solo en la nube.
Lo más valioso para una entrevista no es ningún modelo, sino que sepas explicar
**por qué** cada pieza está como está — y este notebook es justo eso.